### Test Clustern und Splitting


Pseudocode:
Wir bauen eine Funktionsklasse, die Methoden wie Clustern, Cluster view( Graph), und Train/Test/Valid (sind das die drei splits?) ausgibt. Das heißt wir bekommen drei Torch tensoren als ausgabe bei letzter Methode.

Aus Scipy das modul für k-Nearest Neighbour importieren (glaube scipy.cluster.hierachy)

unsere M8 in pandas df umwandeln (schon gemacht).
Alle Dopplungen über Pandas-Maske raus, alle Einträge mit Ident <= 0.3 raus. 

Cluster-Methode:
Logikblock, in dem Schwellenwert angegeben wird über Pandas (df[df["ident"] >= Schwellenwert])
Verschlankung auf Paare (Ident-werte nicht mehr von belang, da jetzt jedes Paar eine Kante erzeugen).
Erzeugen einer Übersetzungstabelle: AMPs in IDS, für spätere rückübersetzung
Aufbauen von Adjavenzmatrix -> Umwandlung is Sparse-Graph (nur Positionen ohne verbindung)
adj_matrix = csr_matrix((weights, (rows, cols)), shape=(num_nodes, num_nodes)) mit weights= np.ones(len(zeilen von Pandas Dataframe)
Clustern mit scipy.cluster.hierachy:
num_clusters, labels = connected_components( csgraph=adj_matrix, directed=False, return_labels=True )
return: ein Pandas-Dataframe 
Cluster-Viewer Methode:
Cluster darstellen in Graph-View (wie in Obsidian)

DataSplitter via Dataloader:
Input: Anteil an Test,Train und Val? (0.8, 0.1, 0.1 z.B) und Modus: Zufällig, oder Größte in Größte Splitgruppe
Assert : sum_anteile == 1.0 sein

Splitt wird über die cluster labels durchgeführt mit if-block. Dabei gibts ein SOrtieralgorithmus. 
Aber das können wir noch später machen.

In [22]:
#==Importe==
%load_ext autoreload 
%autoreload 2
from aguirre.config import Config, DATA_DIR, PROJECT_ROOT
import pandas as pd
import torch 
from scipy.sparse.csgraph import connected_components
from aguirre.data import ClusterSplitter, read_fasta
import numpy as np


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:

# class ClusterSplitter:
#     """Clustert AMP-Sequenzen ueber Zusammenhangskomponenten und
#     verteilt ganze Cluster auf train/val/test."""

#     def __init__(self, pairs: pd.DataFrame, ids, min_ident=0.3):
#         """pairs: DataFrame mit query, target, ident (bereits gefiltert).
#         ids:   alle Sequenz-IDs aus der FASTA, auch partnerlose."""
#         pairs = pairs[pairs["ident"] >= min_ident] 
#         self.pairs = pairs
#         self.ids = ids
#         self.min_ident = min_ident

#         self.labels = None
#         self.threshold = None
#         self.id_to_idx, self.idx_to_id = self._build_index()


#     def _build_index(self):
#         """IDs <-> 0..n-1. Setzt self.id_to_idx und self.idx_to_id."""
#         idx_to_id = list(range(len(sorted(self.ids)))) # Elementnummer zu AMP-IDs
#         id_to_idx = {key: item for item, key in enumerate(sorted(self.ids))} #Dicitonairy für Übersetzung
#         return id_to_idx, idx_to_id

#     def build_graph(self, t):
#         """Schwelle anwenden, Kanten bauen -> csr_matrix. Gibt die Matrix zurueck."""
#         pass

#     def cluster(self, t):
#         """build_graph + connected_components. Setzt self.labels, self.threshold."""
#         pass

#     def summary(self):
#         """Groessenverteilung der Cluster -> DataFrame. Diagnostik."""
#         pass

#     def assign_split(self, ratios=(0.8, 0.1, 0.1), mode="greedy", seed=0):
#         """Ganze Cluster auf die Partitionen verteilen -> DataFrame (id, cluster, split)."""
#         pass

In [3]:
#==Umwandlung in Pandas df==

OUTPATH_M8 = DATA_DIR / "all19K_hits.m8"
COLS = ["query","target","fident","alnlen","qlen","tlen","qcov","tcov","evalue","bits"]
M8 = pd.read_csv(
    filepath_or_buffer= OUTPATH_M8,
    sep = "\t",
    names = COLS,
    usecols= ["query","target","fident","alnlen","qlen","tlen","qcov","tcov"],
    dtype= {"fident":"float32","qcov":"float32","tcov":"float32"},        
)

In [4]:
display(M8.shape)

(17632447, 8)

In [5]:
#==DataFrame mit allen Paaren==
pairs= M8[M8["query"] != M8["target"]].copy()
pairs["ident"] = pairs["fident"] * pairs[["qcov", "tcov"]].max(axis = 1).copy()
display(pairs.shape[0])

#pairs = pairs[pairs["ident"] >= 0.3]
#display(pairs.shape[0])

17613063

In [6]:
#==ids aus Fasta==
OUTPATH_FASTA = DATA_DIR / "all19k_unclustered.faa"
ids, seqs = read_fasta(OUTPATH_FASTA)

print(len(ids), len(seqs))

19384 19384


In [7]:
#==Proof==


cs = ClusterSplitter(pairs=pairs, ids=ids)
print(len(cs.idx_to_id))                      # 19384
print(len(cs.pairs))                          # weniger als vorher

print(type(cs.idx_to_id))
print(list(cs.idx_to_id)[:3])     # was sind die Schlüssel?
print(type(cs.id_to_idx))
print(list(cs.id_to_idx)[:3])
assert all(cs.id_to_idx[cs.idx_to_id[i]] == i for i in range(len(cs.idx_to_id))), "Tja, pech gehabt"

fehlend = (set(cs.pairs["query"]) | set(cs.pairs["target"])) - set(cs.idx_to_id)
print(len(fehlend))                           # muss 0 sein

19384
5904177
<class 'list'>
['AMP_0', 'AMP_1', 'AMP_10']
<class 'dict'>
['AMP_0', 'AMP_1', 'AMP_10']
0


In [8]:
#build graph()
A = cs.build_graph(0.8)
print(A.shape)          # (19384, 19384)
print(A.nnz)            # Anzahl Nicht-Null-Einträge
print((A != A.T).nnz)   # muss 0 sein -> symmetrisch


(19384, 19384)
705090
0


In [9]:
#Cluster

n_comp, labels = cs.cluster(0.6)
print(n_comp, labels.shape, labels.max())

3609 (19384,) 3608


In [23]:
#==Summary==
thresholds = np.linspace(0.3,1, 15, dtype= np.float32).round(2)
rows = []
for t in thresholds:
    cs.cluster(t)
    rows.append(cs.summary())

tabelle = pd.DataFrame(rows).drop(columns="top10")
display(tabelle)

path_tab =PROJECT_ROOT / "wrath_of_god" / "data_tracked" / "cluster_treshold_table.csv"
tabelle.to_csv(path_tab)

,threshold,n_components,largest,largest_share,n_singletons,median
0,0.30,28,19354,0.998452,25,1.0
1,0.35,122,19245,0.992829,107,1.0
2,0.40,402,18856,0.972761,322,1.0
3,0.45,1097,17859,0.921327,885,1.0
4,0.50,1797,16263,0.838991,1394,1.0
5,0.55,2367,14961,0.771822,1832,1.0
6,0.60,3609,12565,0.648215,2885,1.0
7,0.65,4192,11094,0.572328,3328,1.0
8,0.70,5262,8077,0.416684,4234,1.0
9,0.75,6010,5291,0.272957,4821,1.0
